# ArmorVault — Stage 1: PaddleOCR-VL only

This stage deliberately runs only PaddleOCR-VL. It saves the public demo image and its extraction to Google Drive so Stage 2 can run in a fresh runtime. Do not upload real identity documents.

In [ ]:
import subprocess, os
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
work = Path('/content/drive/MyDrive/armorvault-ocr-lab')
work.mkdir(parents=True, exist_ok=True)

In [ ]:
%pip install -q paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%pip install -q 'paddleocr[doc-parser]>=3.7.0,<4' pillow
print('PaddleOCR-VL dependencies installed.')

In [ ]:
from urllib.request import urlretrieve
from PIL import Image
image_path = work / 'public_demo.png'
urlretrieve('https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/paddleocr_vl_demo.png', image_path)
display(Image.open(image_path))

In [ ]:
import json, time
from paddleocr import PaddleOCRVL
paddle_dir = work / 'paddle-output'
paddle_dir.mkdir(exist_ok=True)
started = time.perf_counter()
pipeline = PaddleOCRVL(pipeline_version='v1.6', use_doc_orientation_classify=False, use_doc_unwarping=False)
init_seconds = time.perf_counter() - started
started = time.perf_counter()
count = 0
for result in pipeline.predict(str(image_path)):
    count += 1
    result.save_to_json(save_path=str(paddle_dir))
    result.save_to_markdown(save_path=str(paddle_dir))
inference_seconds = time.perf_counter() - started
parts = []
for path in sorted(paddle_dir.iterdir()):
    if path.suffix.lower() in {'.json', '.md'}:
        parts.append('--- ' + path.name + ' ---\n' + path.read_text(encoding='utf-8', errors='replace'))
(work / 'paddle-extraction.txt').write_text('\n\n'.join(parts), encoding='utf-8')
(work / 'stage1-metadata.json').write_text(json.dumps({'results': count, 'initializationSeconds': init_seconds, 'inferenceSeconds': inference_seconds}, indent=2), encoding='utf-8')
print({'results': count, 'initializationSeconds': round(init_seconds, 2), 'inferenceSeconds': round(inference_seconds, 2)})
print('Saved to Google Drive:', work)

Stage 1 is complete when the final cell says `Saved to Google Drive`. Now choose **Runtime → Disconnect and delete runtime** before opening Stage 2.